# Notebook 2: Baseline Intent Classifier (TF-IDF + Logistic Regression)

This notebook trains a simple baseline intent classifier using TF-IDF features and
Logistic Regression. It is lightweight and runs on any machine without a GPU.

We will use this as the **comparison model** against DistilBERT in Chapter 7.

In [ ]:
import json
import joblib
import numpy as np
import pandas as pd
from sklearn.pipeline import Pipeline
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    classification_report, confusion_matrix, f1_score
)
import matplotlib.pyplot as plt
import seaborn as sns

TRAIN_PATH = '../data/intents_train.jsonl'
TEST_PATH  = '../data/intents_test.jsonl'
MODEL_OUT  = '../models/baseline.joblib'

In [ ]:
def load_jsonl(path):
    rows = []
    with open(path) as f:
        for line in f:
            rows.append(json.loads(line))
    return pd.DataFrame(rows)

train_df = load_jsonl(TRAIN_PATH)
test_df  = load_jsonl(TEST_PATH)
print(f'Train: {len(train_df)} samples | Test: {len(test_df)} samples')
print('\nIntent distribution (train):')
print(train_df['intent'].value_counts())

In [ ]:
model = Pipeline([
    ('tfidf', TfidfVectorizer(ngram_range=(1, 2), min_df=1, sublinear_tf=True)),
    ('clf',   LogisticRegression(max_iter=1000, C=5.0, multi_class='multinomial'))
])

model.fit(train_df['text'], train_df['intent'])
print('Training complete.')

In [ ]:
y_pred = model.predict(test_df['text'])
y_true = test_df['intent']

macro_f1 = f1_score(y_true, y_pred, average='macro')
print(f'Macro F1: {macro_f1:.4f}\n')
print(classification_report(y_true, y_pred))

In [ ]:
labels = sorted(train_df['intent'].unique())
cm = confusion_matrix(y_true, y_pred, labels=labels)

plt.figure(figsize=(9, 7))
sns.heatmap(cm, annot=True, fmt='d', xticklabels=labels, yticklabels=labels, cmap='Blues')
plt.title('Baseline Classifier — Confusion Matrix')
plt.xlabel('Predicted')
plt.ylabel('Actual')
plt.tight_layout()
plt.savefig('../docs/diagrams/baseline_confusion_matrix.png', dpi=150)
plt.show()
print('Saved confusion matrix to docs/diagrams/')

In [ ]:
joblib.dump(model, MODEL_OUT)
print(f'Model saved to {MODEL_OUT}')